In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/06_genai/02_question_classifier.py


%md
# Phase 9 — SQL Generator

This notebook converts a natural-language analytical question
into a structured SQL-generation request.

Responsibilities:

1. Receive a user question.
2. Use the question classifier.
3. Identify the appropriate Gold table.
4. Identify relevant columns.
5. Construct an LLM prompt.
6. Generate structured SQL.
7. Return SQL and visualization metadata.

This notebook DOES NOT execute SQL.

SQL execution is handled separately to preserve security boundaries.

In [0]:
from pyspark.sql import functions as F
import json
import re

print("SQL Generator initialized.")

In [0]:
GOLD_TABLE_METADATA = {

    "genai_copilot.gold.monthly_sales": {
        "purpose": "Monthly revenue, cost, profit and order metrics.",
        "columns": [
            "year",
            "month",
            "month_name",
            "total_orders",
            "total_quantity",
            "total_revenue",
            "total_cost",
            "total_profit",
            "profit_margin"
        ]
    },

    "genai_copilot.gold.region_sales": {
        "purpose": "Revenue, cost, profit and order metrics by region.",
        "columns": [
            "region",
            "total_orders",
            "total_quantity",
            "total_revenue",
            "total_cost",
            "total_profit",
            "profit_margin"
        ]
    },

    "genai_copilot.gold.customer_metrics": {
        "purpose": "Customer-level sales and profitability metrics.",
        "columns": [
            "customer_id",
            "customer_name",
            "total_orders",
            "total_quantity",
            "total_revenue",
            "total_cost",
            "total_profit",
            "profit_margin"
        ]
    },

    "genai_copilot.gold.product_metrics": {
        "purpose": "Product-level sales and profitability metrics.",
        "columns": [
            "product_id",
            "product_name",
            "category",
            "total_orders",
            "total_quantity",
            "total_revenue",
            "total_cost",
            "total_profit",
            "profit_margin"
        ]
    },

    "genai_copilot.gold.category_metrics": {
        "purpose": "Category-level sales and profitability metrics.",
        "columns": [
            "category",
            "total_orders",
            "total_quantity",
            "total_revenue",
            "total_cost",
            "total_profit",
            "profit_margin"
        ]
    }
}

print("Gold table metadata loaded.")

for table_name, metadata in GOLD_TABLE_METADATA.items():
    print(f"\n{table_name}")
    print(metadata["purpose"])

In [0]:
def select_gold_table(question, intent):
    """
    Select the most appropriate Gold table
    based on question keywords and intent.
    """

    q = question.lower()

    # Customer questions
    if any(
        keyword in q
        for keyword in [
            "customer",
            "customers",
            "client"
        ]
    ):
        return "genai_copilot.gold.customer_metrics"

    # Product questions
    if any(
        keyword in q
        for keyword in [
            "product",
            "products"
        ]
    ):
        return "genai_copilot.gold.product_metrics"

    # Category questions
    if any(
        keyword in q
        for keyword in [
            "category",
            "categories"
        ]
    ):
        return "genai_copilot.gold.category_metrics"

    # Region questions
    if any(
        keyword in q
        for keyword in [
            "region",
            "regions",
            "country",
            "japan",
            "usa",
            "france",
            "germany"
        ]
    ):
        return "genai_copilot.gold.region_sales"

    # Trend questions
    if intent == "trend":
        return "genai_copilot.gold.monthly_sales"

    # Visualization involving time
    if (
        "monthly" in q
        or "month" in q
        or "over time" in q
    ):
        return "genai_copilot.gold.monthly_sales"

    # Default analytical table
    return "genai_copilot.gold.monthly_sales"

In [0]:
table_selection_tests = [

    ("What is monthly revenue?", "trend"),

    ("Which region generated the highest revenue?", "ranking"),

    ("What are the top 10 customers by revenue?", "ranking"),

    ("Which product generated the most profit?", "ranking"),

    ("Which category has the highest profit margin?", "ranking"),

]

for question, intent in table_selection_tests:

    table = select_gold_table(
        question,
        intent
    )

    print("=" * 70)
    print("Question:", question)
    print("Selected table:", table)

In [0]:
def build_schema_context(table_name):
    """
    Build compact schema context for the LLM.

    We deliberately send metadata rather than the entire dataset.
    """

    if table_name not in GOLD_TABLE_METADATA:
        raise ValueError(
            f"Unknown Gold table: {table_name}"
        )

    metadata = GOLD_TABLE_METADATA[table_name]

    context = {
        "table_name": table_name,
        "purpose": metadata["purpose"],
        "columns": metadata["columns"]
    }

    return context


schema_context = build_schema_context(
    "genai_copilot.gold.region_sales"
)

print(
    json.dumps(
        schema_context,
        indent=2
    )
)

In [0]:
SQL_BUSINESS_RULES = [

    "Only generate read-only SELECT statements.",

    "Use only approved Gold tables.",

    "Do not access Bronze or Silver tables unless explicitly allowed.",

    "Do not generate INSERT, UPDATE, DELETE, DROP, ALTER, CREATE or TRUNCATE.",

    "Do not generate multiple SQL statements.",

    "Do not use filesystem commands.",

    "Do not use external commands.",

    "Do not access secrets.",

    "Use explicit column names.",

    "Use aggregation functions when required.",

    "Use ORDER BY for ranking questions.",

    "Use LIMIT for top-N questions.",

    "Avoid SELECT * unless absolutely necessary.",

    "Return only SQL-compatible syntax.",

]

print("SQL generation rules loaded.")

In [0]:
def build_sql_generation_prompt(
    question,
    intent,
    table_name
):
    """
    Build a controlled prompt for SQL generation.
    """

    schema_context = build_schema_context(
        table_name
    )

    rules = "\n".join(
        f"- {rule}"
        for rule in SQL_BUSINESS_RULES
    )

    prompt = f"""
You are a senior data analyst working with Databricks SQL.

Your task is to generate a read-only analytical SQL query.

USER QUESTION:
{question}

CLASSIFIED INTENT:
{intent}

APPROVED TABLE:
{table_name}

TABLE PURPOSE:
{schema_context["purpose"]}

AVAILABLE COLUMNS:
{", ".join(schema_context["columns"])}

BUSINESS RULES:
{rules}

Return JSON with exactly these fields:

{{
  "intent": "...",
  "table": "...",
  "sql": "...",
  "chart": "...",
  "explanation": "..."
}}

Allowed chart values:

- line
- bar
- pie
- scatter
- table

Do not execute the SQL.

Do not include markdown fences.
"""

    return prompt.strip()

In [0]:
prompt = build_sql_generation_prompt(
    question="Which region generated the highest revenue?",
    intent="ranking",
    table_name="genai_copilot.gold.region_sales"
)

print(prompt)

In [0]:
def generate_sql_fallback(
    question,
    intent,
    table_name
):
    """
    Deterministic fallback SQL generator.

    This allows the project to operate without
    a paid external LLM service.

    The LLM version can replace this function later.
    """

    q = question.lower()

    # --------------------------------------------------------
    # TOTAL REVENUE
    # --------------------------------------------------------

    if "total revenue" in q:

        return {
            "intent": "aggregation",
            "table": table_name,
            "sql": f"""
SELECT
    SUM(total_revenue) AS total_revenue
FROM {table_name}
""".strip(),
            "chart": "table",
            "explanation": "Calculates total revenue."
        }

    # --------------------------------------------------------
    # REGION RANKING
    # --------------------------------------------------------

    if (
        "region" in q
        and (
            "highest" in q
            or "top" in q
            or "most" in q
        )
    ):

        return {
            "intent": "ranking",
            "table": table_name,
            "sql": f"""
SELECT
    region,
    total_revenue
FROM {table_name}
ORDER BY total_revenue DESC
LIMIT 10
""".strip(),
            "chart": "bar",
            "explanation": (
                "Ranks regions by total revenue "
                "in descending order."
            )
        }

    # --------------------------------------------------------
    # CUSTOMER RANKING
    # --------------------------------------------------------

    if (
        "customer" in q
        and (
            "top" in q
            or "highest" in q
            or "most" in q
        )
    ):

        return {
            "intent": "ranking",
            "table": table_name,
            "sql": f"""
SELECT
    customer_id,
    customer_name,
    total_revenue,
    total_profit
FROM {table_name}
ORDER BY total_revenue DESC
LIMIT 10
""".strip(),
            "chart": "bar",
            "explanation": (
                "Ranks customers by total revenue."
            )
        }

    # --------------------------------------------------------
    # MONTHLY REVENUE
    # --------------------------------------------------------

    if (
        "monthly revenue" in q
        or "revenue by month" in q
        or "monthly sales" in q
    ):

        return {
            "intent": "trend",
            "table": table_name,
            "sql": f"""
SELECT
    year,
    month,
    month_name,
    total_revenue
FROM {table_name}
ORDER BY year, month
""".strip(),
            "chart": "line",
            "explanation": (
                "Shows monthly revenue over time."
            )
        }

    # --------------------------------------------------------
    # CATEGORY PROFIT MARGIN
    # --------------------------------------------------------

    if (
        "category" in q
        and "profit margin" in q
    ):

        return {
            "intent": "ranking",
            "table": table_name,
            "sql": f"""
SELECT
    category,
    profit_margin
FROM {table_name}
ORDER BY profit_margin DESC
LIMIT 10
""".strip(),
            "chart": "bar",
            "explanation": (
                "Ranks product categories by profit margin."
            )
        }

    # --------------------------------------------------------
    # FALLBACK
    # --------------------------------------------------------

    return {
        "intent": intent,
        "table": table_name,
        "sql": None,
        "chart": "table",
        "explanation": (
            "No deterministic SQL template was found. "
            "LLM generation is required."
        )
    }

In [0]:
# Make sure the classifier function exists
# from the previous notebook/session.

def generate_sql_request(question):
    """
    Main SQL generation entry point.
    """

    classification = classify_question(
        question
    )

    # --------------------------------------------------------
    # RAG QUESTIONS
    # --------------------------------------------------------

    if classification["intent"] == "rag":

        return {
            "question": question,
            "route": "rag",
            "classification": classification,
            "sql_required": False,
            "sql": None
        }

    # --------------------------------------------------------
    # HYBRID QUESTIONS
    # --------------------------------------------------------

    if classification["intent"] == "hybrid":

        return {
            "question": question,
            "route": "hybrid",
            "classification": classification,
            "sql_required": True,
            "sql": None,
            "message": (
                "Hybrid processing requires SQL generation "
                "and document retrieval."
            )
        }

    # --------------------------------------------------------
    # UNSUPPORTED
    # --------------------------------------------------------

    if classification["intent"] == "unsupported":

        return {
            "question": question,
            "route": "unsupported",
            "classification": classification,
            "sql_required": False,
            "sql": None
        }

    # --------------------------------------------------------
    # SQL
    # --------------------------------------------------------

    table_name = select_gold_table(
        question,
        classification["intent"]
    )

    sql_result = generate_sql_fallback(
        question,
        classification["intent"],
        table_name
    )

    return {
        "question": question,
        "route": "sql",
        "classification": classification,
        "table": table_name,
        "sql_required": True,
        "generation": sql_result
    }

In [0]:
result = generate_sql_request(
    "Which region generated the highest revenue?"
)

print(
    json.dumps(
        result,
        indent=2,
        default=str
    )
)

In [0]:
questions = [

    "What is total revenue?",

    "Which region generated the highest revenue?",

    "What are the top 10 customers by revenue?",

    "What is monthly revenue?",

    "Which category has the highest profit margin?",

    "What is the discount policy?",

    "Compare Japanese sales with the company discount policy."

]

for question in questions:

    print("=" * 80)
    print("QUESTION:", question)

    result = generate_sql_request(
        question
    )

    print("ROUTE:", result["route"])

    if result.get("table"):
        print("TABLE:", result["table"])

    if result.get("generation"):
        print("\nSQL:")
        print(result["generation"]["sql"])

        print("\nCHART:")
        print(result["generation"]["chart"])

In [0]:
def validate_generation_structure(result):

    required_fields = [
        "question",
        "route"
    ]

    for field in required_fields:

        if field not in result:
            return False

    # SQL route must contain generation
    if result["route"] == "sql":

        if "generation" not in result:
            return False

        generation = result["generation"]

        required_generation_fields = [
            "intent",
            "table",
            "sql",
            "chart",
            "explanation"
        ]

        for field in required_generation_fields:

            if field not in generation:
                return False

    return True

In [0]:
generation_tests = [
    "What is total revenue?",
    "Which region generated the highest revenue?",
    "What are the top 10 customers by revenue?",
    "What is monthly revenue?",
    "What is the discount policy?",
]

for question in generation_tests:

    result = generate_sql_request(
        question
    )

    valid = validate_generation_structure(
        result
    )

    print(
        f"{'PASS' if valid else 'FAIL'} | {question}"
    )

In [0]:
ALLOWED_GOLD_TABLES = set(
    GOLD_TABLE_METADATA.keys()
)


def check_allowed_table(
    table_name
):
    return table_name in ALLOWED_GOLD_TABLES


for table_name in ALLOWED_GOLD_TABLES:

    print(
        table_name,
        "->",
        check_allowed_table(table_name)
    )

In [0]:
question = (
    "Which region generated the highest revenue?"
)

classification = classify_question(
    question
)

table_name = select_gold_table(
    question,
    classification["intent"]
)

schema_context = build_schema_context(
    table_name
)

generation = generate_sql_fallback(
    question,
    classification["intent"],
    table_name
)

print("=" * 80)
print("END-TO-END SQL GENERATION TEST")
print("=" * 80)

print("\nQuestion:")
print(question)

print("\nIntent:")
print(classification["intent"])

print("\nSelected Table:")
print(table_name)

print("\nSchema:")
print(schema_context)

print("\nGenerated SQL:")
print(generation["sql"])

print("\nChart:")
print(generation["chart"])

print("\nExplanation:")
print(generation["explanation"])

This one notebook demonstrates:

Natural-language analytics
Intent-aware routing
Schema-aware SQL generation
Gold-layer utilization
Controlled LLM prompting
Read-only SQL design
Allowlisted tables
Visualization recommendation
Separation of generation and execution
Free-edition-friendly architecture

Most importantly, we're avoiding the weak portfolio pattern of:

"Send question directly to ChatGPT and execute whatever SQL comes back."

Instead we're building:

Question
   ↓
Classification
   ↓
Schema intelligence
   ↓
Controlled generation
   ↓
Security validation
   ↓
Execution
   ↓
Result validation
   ↓
Explanation

That is much closer to a production-oriented GenAI data application.

Important architecture decision

We are intentionally building this in two stages.

Stage 1 — Free/local deterministic generator
Question
   ↓
Classifier
   ↓
Table selection
   ↓
SQL template

This allows you to develop and test the entire pipeline without requiring a paid LLM endpoint.

Stage 2 — LLM-powered generator

Later we'll add:

Question
   ↓
Classifier
   ↓
Schema Retrieval
   ↓
LLM
   ↓
Structured SQL
   ↓
SQL Validator

The LLM will not be allowed to execute anything.

In [0]:
generate_sql_request(
    "Which region generated the highest revenue?"
)

In [0]:
validate_generation_structure(
    generate_sql_request(
        "Which region generated the highest revenue?"
    )
)